# Inside the conv projector: how one audio token is made

`PROJECTOR = "conv"` (`T.ConvAudioProjector` in [`train.py`](train.py)) is the only thing between the mel spectrogram and
the LLM. This notebook loads a **trained** conv projector, picks one held-out utterance and **one audio token**, and
follows that token through every layer:

| § | question |
|---|---|
| 0 | Setup: which run, which utterance, which token |
| 1 | What are the layers, and how many positions does each one have per token? |
| 2 | Which mel frames can reach the token? (its family tree / receptive field) |
| 3 | What does the token look like after each layer? |
| 4 | Which input frames and mel bins does the token *actually* depend on? |
| 5 | Where does the finished token land in the LLM's embedding space? |
| 6 | What does the first conv layer look for? |

With `STACK = 8` and the 230M LLM (`H = 1024`) the projector is:

```
normalised log-mel        (8·T frames × 80)                     100 Hz   8 positions per token
  conv1  k=3 s=2 : 3 frames × 80    → 512,  GELU                  50 Hz   4
  conv2  k=3 s=2 : 3 pos   × 512    → 1024, GELU                  25 Hz   2
  conv3  k=3 s=2 : 3 pos   × 1024   → 1024, GELU                12.5 Hz   1   ← one position per audio token
  LayerNorm(1024)
  fc1    1024 → 1024, GELU                                                  ┐ the "2-layer MLP", applied to each
  out    1024 → 1024                                                        ┘ token on its own
  ──► audio token (1024-d), placed in the LLM input where a word embedding would go
```

So the "conv layers + 2 MLP layers" are really two parts with different jobs. The **convs** are the only layers that mix
neighbouring frames, and they cut 100 Hz down to 12.5 Hz. The **MLP** is per-token: it never sees a neighbour and only
reshapes each token's vector into something the LLM can read. Everything after that (context across tokens) is the
LLM's job, with LoRA.

Runs on CPU in about a minute and leaves the GPU to training. Every knob is in the `PARAMETERS` block below.

## 0. Setup

In [ ]:
import json, os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "train.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import soundfile as sf
from matplotlib.patches import Rectangle
from IPython.display import Audio, display
from transformers import AutoConfig

import train as T

# ═══════════════════════════ PARAMETERS: edit, then Run All ═══════════════════════════
RUN = ""                     # run dir trained with PROJECTOR="conv"; "" = the newest finished one (has proj.pt)
SAMPLE_SPLIT = "dev-clean"   # held out: never trained on
SAMPLE_INDEX = 10            # utterance index in that split, sorted by id
TOKEN = None                 # audio token to follow; None = the loudest one
N_PROBES = 64                # random probes per token for the whole-utterance influence map (section 4c)
# ════════════════════════════════════════════════════════════════════════════════════════

if not RUN:
    finished = sorted(Path("runs").glob("*/proj.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    conv_runs = [p.parent for p in finished if json.load(open(p.parent / "config.json")).get("PROJECTOR") == "conv"]
    assert conv_runs, "no finished run with PROJECTOR='conv' in runs/"
    RUN = conv_runs[0]
RUN = Path(RUN)
T.apply_config(json.load(open(RUN / "config.json")))   # STACK, N_MELS, MVN split, ... exactly as trained
assert T.PROJECTOR == "conv", f"{RUN} was trained with PROJECTOR={T.PROJECTOR!r}"
torch.set_num_threads(4)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})

S, M, FPS = T.STACK, T.N_MELS, T.SR / T.HOP
H = AutoConfig.from_pretrained(T.MODEL).hidden_size
proj = T.ConvAudioProjector(M, S, H)
proj.load_state_dict({k: v.float() for k, v in torch.load(RUN / "proj.pt", map_location="cpu").items()})
proj.eval().requires_grad_(False)

mvn = torch.load(T.mvn_cache_path())
tok = T.AutoTokenizer.from_pretrained(T.MODEL)
sample = sorted(T.load_split(SAMPLE_SPLIT), key=lambda it: it["id"])[SAMPLE_INDEX]
x = T.SpeechDataset([sample], mvn, tok)[0]["audio"]   # (n_tok, STACK * N_MELS): exactly what training feeds the projector
mel = x.reshape(-1, M)                                # the same numbers as continuous frames
n_tok = len(x)
if TOKEN is None:                                     # loudest token, not at the edges
    TOKEN = int(x.mean(1)[1:-1].argmax()) + 1
j = TOKEN

print(f"run    : {RUN}  (STACK={S}, N_MELS={M}, H={H}, {FPS / S:g} Hz tokens = {1000 * S / FPS:.0f} ms each)")
print(f"sample : {SAMPLE_SPLIT}[{SAMPLE_INDEX}] {sample['id']}, {sample['sec']:.2f} s -> {len(mel)} frames -> {n_tok} audio tokens")
print(f"text   : {sample['text']}")
print(f"token  : j = {j}, its own patch = frames {S * j}..{S * j + S - 1} = {S * j / FPS:.2f}-{(S * j + S) / FPS:.2f} s")
display(Audio(sf.read(sample["path"], dtype="float32")[0], rate=T.SR))

## 1. The layers, one position count at a time

`ConvAudioProjector.forward` writes each conv as `unfold` + `Linear` (a GEMM, which avoids slow ROCm conv kernels for
variable lengths). `trace()` below repeats that forward pass and keeps every intermediate, and checks the result against
the real module. The second check shows the trick is an ordinary `nn.Conv1d(kernel=3, stride=2, padding=1)`: a window
is flattened as `(channel, tap)`, so `Linear.weight.view(out, in, 3)` *is* the conv kernel.

In [ ]:
N_CONV = len(proj.convs)                              # log2(STACK): 3 for STACK=8
CONVS = [f"conv{i}" for i in range(1, N_CONV + 1)]
STAGES = ["mel", *CONVS, "LayerNorm", "fc1+GELU", "out"]
STRIDE = {"mel": 1, **{c: 2 ** i for i, c in enumerate(CONVS, 1)}, "LayerNorm": S, "fc1+GELU": S, "out": S}  # frames per position

def trace(stacked):
    """Same maths as ConvAudioProjector.forward, keeping every stage: {stage: (positions, channels)}."""
    acts = {"mel": stacked.reshape(-1, proj.n_mels)}
    h = acts["mel"]
    for i, conv_linear in enumerate(proj.convs, 1):
        windows = F.pad(h, (0, 0, 1, 1)).unfold(0, 3, 2).reshape(h.shape[0] // 2, -1)   # (positions/2, in * 3)
        h = acts[f"conv{i}"] = proj.act(conv_linear(windows))
    acts["LayerNorm"] = proj.norm(h)
    acts["fc1+GELU"] = proj.act(proj.fc1(acts["LayerNorm"]))
    acts["out"] = proj.out(acts["fc1+GELU"])
    return acts

acts = trace(x)
y = acts["out"]
assert torch.allclose(y, proj(x), atol=1e-5)

h = mel.T[None]                                       # (1, N_MELS, frames)
for conv_linear in proj.convs:
    h = F.gelu(F.conv1d(h, conv_linear.weight.view(conv_linear.out_features, -1, 3), conv_linear.bias, stride=2, padding=1))
assert torch.allclose(h[0].T, acts[CONVS[-1]], atol=1e-4)
print("trace() == proj(x), and the unfold+Linear stages == nn.Conv1d(k=3, s=2, p=1)\n")

layer = {**dict(zip(CONVS, proj.convs)), "LayerNorm": proj.norm, "fc1+GELU": proj.fc1, "out": proj.out}
print(f"{'stage':<11}{'shape':>14}{'rate Hz':>9}{'pos/token':>10}{'sees':>16}{'params':>12}")
for s in STAGES:
    p = sum(w.numel() for w in layer[s].parameters()) if s in layer else 0
    sees = "-" if s == "mel" else "3 frames" if s == "conv1" else f"3 conv{int(s[4:]) - 1} pos" if s in CONVS else "1 token"
    print(f"{s:<11}{str(tuple(acts[s].shape)):>14}{FPS / STRIDE[s]:>9.1f}{S // STRIDE[s]:>10}{sees:>16}{p:>12,}")
n_conv = sum(w.numel() for c in proj.convs for w in c.parameters())
n_all = sum(w.numel() for w in proj.parameters())
print(f"\ntotal {n_all:,} params: convs {n_conv / n_all:.0%}, LayerNorm + MLP {1 - n_conv / n_all:.0%}")

## 2. The token's family tree: which frames can reach it?

Each conv position reads 3 positions of the layer below, centred on position `2p`. Following that down from token `j`
gives its **family tree**. The receptive field is `2·STACK − 1` frames, `STACK·j − (STACK−1) … STACK·j + STACK−1`. That
is the token's **own patch plus the last `STACK−1` frames of the previous patch**. It is lopsided: token `j` is centred
on the *first* frame of its own patch, so it looks back `STACK−1` frames (70 ms at `STACK=8`) and never past the end of
its own patch.
Neighbouring tokens therefore share frames, which the `linear` / `mlp` projectors never do.

The tree does not reach every frame equally (bottom panel). A stride-2, kernel-3 conv reads even positions once and
odd positions twice, so after 3 stages the path count is a **comb** (1-3 paths per frame) and not a smooth bump.
Edges shared by both trees look grey. Section 4 shows that the trained token's sensitivity still follows this comb.

In [ ]:
def lineage(j):
    """{stage: positions in token j's family tree} and {frame: number of distinct paths frame -> token j}."""
    levels = ["mel", *CONVS][::-1]                     # top conv first
    pos = {levels[0]: {j: 1}}
    for upper, lower in zip(levels[:-1], levels[1:]):
        n_lower = len(acts[lower])
        nxt = {}
        for p, count in pos[upper].items():
            for q in (2 * p - 1, 2 * p, 2 * p + 1):
                if 0 <= q < n_lower:                    # outside = the zero padding of that stage
                    nxt[q] = nxt.get(q, 0) + count
        pos[lower] = nxt
    return pos

tree = lineage(j)
rf = sorted(tree["mel"])
print(f"token {j}: receptive field frames {rf[0]}..{rf[-1]} ({len(rf)} frames, {len(rf) / FPS * 1000:.0f} ms); "
      f"own patch {S * j}..{S * j + S - 1}")
print("paths per frame:", [tree["mel"][f] for f in rf])

LEVELS = ["mel", *CONVS, "out"]                       # rows of the diagram; "out" = the finished token
trees = {t: {**lineage(t), "out": {t: 1}} for t in (j, j + 1) if t < n_tok}
lo_f, hi_f = S * (j - 2), min(S * (j + 3), len(mel))      # frames shown: patches j-2 .. j+2
fig, (ax, ax2, ax3) = plt.subplots(3, 1, figsize=(13, 8.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.4, 1]})
COLOR = {j: "C0", j + 1: "C1"}
for t, tr in trees.items():                            # edges: each position -> the 3 below it
    z = 2 if t == j else 1
    for lvl in range(N_CONV, 0, -1):
        up, down = LEVELS[lvl], LEVELS[lvl - 1]
        for p in tr[up]:
            for q in (2 * p - 1, 2 * p, 2 * p + 1):
                if q in tr[down]:
                    ax.plot([p * STRIDE[up], q * STRIDE[down]], [lvl, lvl - 1], color=COLOR[t], lw=1.1, alpha=0.6, zorder=z)
    ax.plot([t * S, t * S], [N_CONV, N_CONV + 1], color=COLOR[t], lw=2.2, zorder=z)
for lvl, name in enumerate(LEVELS):
    st = STRIDE[name]
    ps = [p for p in range(len(acts[name])) if lo_f - st <= p * st < hi_f]
    ax.scatter([p * st for p in ps], [lvl] * len(ps), s=22, color="0.75", zorder=3)
    for t in sorted(trees, reverse=True):
        mine = [p for p in ps if p in trees[t][name]]
        ax.scatter([p * st for p in mine], [lvl] * len(mine), s=40, color=COLOR[t], zorder=4)
ax.set_yticks(range(N_CONV + 2), ["mel frames\n100 Hz", *[f"{c}\n{FPS / STRIDE[c]:g} Hz" for c in CONVS], "LN + MLP\n= token"])
ax.set_title(f"family trees of token {j} (blue) and token {j + 1} (orange): every frame that can reach them", loc="left")
for k in range(j - 2, j + 4):
    ax.axvline(k * S - 0.5, color="0.85", lw=0.8, zorder=0)

ax2.imshow(mel[lo_f:hi_f].T, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3,
           extent=[lo_f - 0.5, hi_f - 0.5, 0, M])
ax2.add_patch(Rectangle((rf[0] - 0.5, 0.5), len(rf), M - 1, fill=False, ec="C0", lw=2, ls="--"))
ax2.add_patch(Rectangle((S * j - 0.5, 2), S, M - 4, fill=False, ec="k", lw=1.5))
ax2.set_ylabel("mel bin"); ax2.set_title("normalised log-mel: own patch (black) and receptive field (blue dashed)", loc="left", fontsize=9)

counts = np.array([tree["mel"].get(f, 0) for f in range(lo_f, hi_f)])
ax3.bar(range(lo_f, hi_f), counts, color="0.6", width=0.8)
ax3.set(ylabel="paths", xlabel=f"frame (10 ms); patch boundaries every {S} frames")
ax3.set_title(f"distinct paths from each frame to token {j}", loc="left", fontsize=9)
for a in (ax2, ax3):
    for k in range(j - 2, j + 4):
        a.axvline(k * S - 0.5, color="0.5", lw=0.6)
plt.tight_layout(); plt.show()

## 3. The token through every layer

Every stage over ~1 s around token `j`, on one shared time axis. Each column is one position, as wide as the frames it
stands for (`conv1` = 20 ms, `conv2` = 40 ms, `conv3` and later = 80 ms). The outlined columns are token `j`'s family
tree. Each column is drawn centred on the frame its tree is centred on, so token `j`'s column
straddles the boundary between patches `j−1` and `j` (dotted lines), the same lopsidedness as in section 2. Channels are
sorted by their mean activation over the utterance so that structure is visible. After the convs
(GELU, mostly ≥ 0) the colour is sequential. LayerNorm and the outputs are signed, so they use a diverging map.

In [ ]:
lo_t, hi_t = max(0, j - 6), min(n_tok, j + 7)          # tokens shown
fig, axes = plt.subplots(len(STAGES), 1, figsize=(13, 15), sharex=True,
                         gridspec_kw={"height_ratios": [1.2, 1, 1, 1, 1, 1, 1]})
for ax, s in zip(axes, STAGES):
    st = STRIDE[s]
    a = acts[s]
    order = a.mean(0).argsort(descending=True)
    p0, p1 = lo_t * S // st, hi_t * S // st
    view = a[p0:p1, order].T
    signed = s in ("mel", "LayerNorm", "out")
    lim = float(view.abs().quantile(0.99)) if signed else float(view.quantile(0.99))
    # position p is centred on frame p * stride (mel frames: on themselves)
    left = (p0 * st - (st / 2 if s != "mel" else 0)) / FPS
    im = ax.imshow(view, aspect="auto", origin="lower", interpolation="nearest",
                   cmap="RdBu_r" if signed else "magma", vmin=-lim if signed else 0, vmax=lim,
                   extent=[left, left + (p1 - p0) * st / FPS, 0, a.shape[1]])
    fam = sorted(tree.get(s, {j: 1}))                  # LayerNorm / MLP: just token j
    c = (fam[0] * st - (st / 2 if s != "mel" else 0)) / FPS
    ax.add_patch(Rectangle((c, 0), len(fam) * st / FPS, a.shape[1], fill=False, ec="C0", lw=2))
    ax.set_ylabel(f"{s}\n{a.shape[1]} ch", fontsize=8)
    plt.colorbar(im, ax=ax, pad=0.01, fraction=0.02)
for t in range(lo_t, hi_t + 1):                        # token (patch) boundaries
    for ax in axes:
        ax.axvline(t * S / FPS, color="0.6", lw=0.6, ls=":")
axes[0].set_title(f"activations around token {j} ({S * j / FPS:.2f} s); outlined = token {j}'s family tree; dotted = patch edges", loc="left")
axes[-1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

### 3b. Token `j`'s vector, stage by stage

Each row is one stage at token `j`'s position (for `conv1` / `conv2`, the positions in its tree). Columns:

* **norm**: vector size.
* **active**: fraction of units with GELU output > 0.05. The convs and `fc1` decide which features are "on" here.
* **cos(j−1)** / **cos(j+1)**: cosine similarity to the neighbouring token at the same stage. If it is high, neighbours
  look alike at that stage. A drop through the MLP means it spreads them apart.

In [ ]:
def stage_vec(s, t):
    """Stage s at token t: conv1/conv2 have several positions per token -> the ones centred in its patch, concatenated."""
    st = STRIDE[s]
    return acts[s][t * S // st:(t + 1) * S // st].flatten()

print(f"{'stage':<11}{'dim':>6}{'norm':>8}{'active':>8}{'cos(j-1)':>10}{'cos(j+1)':>10}")
for s in STAGES:
    v = stage_vec(s, j)
    active = f"{(v > 0.05).float().mean():.0%}" if s.startswith(("conv", "fc1")) else ""
    cos = [F.cosine_similarity(v, stage_vec(s, t), dim=0).item() if 0 <= t < n_tok else float("nan") for t in (j - 1, j + 1)]
    print(f"{s:<11}{len(v):>6}{v.norm():>8.2f}{active:>8}{cos[0]:>10.2f}{cos[1]:>10.2f}")

# cosine similarity between all tokens at three stages
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, s in zip(axes, ["conv3", "LayerNorm", "out"]):
    a = F.normalize(acts[s], dim=1)
    im = ax.imshow((a @ a.T).numpy(), cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
    ax.axhline(j, color="k", lw=0.5); ax.axvline(j, color="k", lw=0.5)
    ax.set(title=f"cos(token, token) after {s}", xlabel="token", ylabel="token")
plt.colorbar(im, ax=axes, fraction=0.02)
plt.show()

## 4. What does token `j` actually depend on?

Section 2 shows which frames *can* reach the token. Training decides how much each one *does*.

**4a. Exact Jacobian.** `∂ token_j / ∂ mel[frame, bin]` is a `1024 × (frames × 80)` matrix. Its column norms say how
strongly each input number moves the token. The token only depends on patches `j−1` and `j`, so a 3-patch crop gives
exactly the same token (checked below), and the Jacobian of that crop is cheap.

In [ ]:
from torch.func import jacrev

c0 = max(0, j - 1)                                     # crop = patches c0 .. j+1, contains the whole receptive field
crop = x[c0:j + 2]
assert torch.allclose(proj(crop)[j - c0], y[j], atol=1e-5), "crop changes the token"
J = jacrev(lambda c: proj(c)[j - c0])(crop)            # (H, crop tokens, STACK * N_MELS)
J = J.reshape(H, -1, M)                                # (H, crop frames, N_MELS)
infl = J.norm(dim=0)                                   # (crop frames, N_MELS)
f0 = c0 * S
frames = torch.arange(f0, f0 + len(infl))
per_frame = infl.norm(dim=1)
print(f"influence outside frames {rf[0]}..{rf[-1]}: {per_frame[(frames < rf[0]) | (frames > rf[-1])].abs().max():.2e} (must be 0)")
own = per_frame[(frames >= S * j) & (frames < S * j + S)].pow(2).sum() / per_frame.pow(2).sum()
print(f"share of squared influence from its own patch: {own:.0%}, from the previous patch: {1 - own:.0%}")

fig, axes = plt.subplots(2, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [3, 1], "height_ratios": [2, 1]})
ext = [frames[0] - 0.5, frames[-1] + 0.5, 0, M]
im = axes[0, 0].imshow(infl.T, origin="lower", aspect="auto", cmap="magma", extent=ext)
axes[0, 0].add_patch(Rectangle((S * j - 0.5, 0.5), S, M - 1, fill=False, ec="w", lw=1.5))
axes[0, 0].set(ylabel="mel bin", title=f"||∂ token {j} / ∂ mel[frame, bin]||   (white = own patch)")
plt.colorbar(im, ax=axes[0, 0], pad=0.01)
axes[0, 1].plot(infl.norm(dim=0), np.arange(M), color="C0")
axes[0, 1].set(title="per mel bin", xlabel="influence", ylim=(0, M))
paths = np.array([tree["mel"].get(f, 0) for f in frames.tolist()], dtype=float)
axes[1, 0].plot(frames, per_frame / per_frame.max(), "o-", ms=4, lw=2, color="C0", label="learned influence")
axes[1, 0].plot(frames, paths / paths.max(), "s--", ms=4, lw=1.2, color="0.55", label="paths in the tree (architecture only)")
axes[1, 0].axvspan(S * j - 0.5, S * j + S - 0.5, color="0.9", zorder=0)
axes[1, 0].set(xlabel="frame", ylabel="relative to max", title="per frame (grey band = own patch)"); axes[1, 0].legend(fontsize=8)
axes[1, 1].axis("off")
plt.tight_layout(); plt.show()

**4b. Occlusion.** Replace one frame with zeros (after MVN, 0 is the "average frame") and measure how far each token
moves: `‖token' − token‖ / ‖token‖`. Unlike the Jacobian, this measures a real, large change. Each token only reacts to
its own window, and neighbouring windows overlap by `STACK − 1` frames.

In [ ]:
lo_f, hi_f = S * max(0, j - 2), min(len(mel), S * (j + 3))
show = [t for t in (j - 1, j, j + 1) if 0 <= t < n_tok]
change = {t: [] for t in show}
for f in range(lo_f, hi_f):
    m2 = mel.clone(); m2[f] = 0
    y2 = proj(m2.reshape(-1, S * M))
    for t in show:
        change[t].append(((y2[t] - y[t]).norm() / y[t].norm()).item())

fig, ax = plt.subplots(figsize=(13, 3.2))
for t in show:
    ax.plot(range(lo_f, hi_f), change[t], "o-", ms=3.5, lw=2 if t == j else 1.4,
            color={j - 1: "C1", j: "C0", j + 1: "C2"}[t], label=f"token {t}")
for k in range(lo_f // S, hi_f // S + 1):
    ax.axvline(k * S - 0.5, color="0.8", lw=0.8, zorder=0)
ax.set(xlabel="zeroed frame", ylabel="relative change of token", title="occlusion: zero one frame, see which tokens move")
ax.legend(); plt.show()

**4c. The whole utterance at once.** The same influence for *every* token gives a token × frame map. Exact Jacobians
for all tokens would be slow, so this uses random probes: `E_v ‖vᵀ J‖² = ‖J‖²_F`, with `N_PROBES` Gaussian `v`. Tokens two
apart have disjoint receptive fields, so even and odd tokens can each be probed in one backward pass. Expect a staircase
band: the token rate is one eighth of the frame rate, and each step is 15 frames wide.

In [ ]:
torch.manual_seed(0)
band = torch.zeros(n_tok, len(mel))
xg = x.clone().requires_grad_(True)
with torch.enable_grad():
    for parity in (0, 1):
        sel = torch.arange(parity, n_tok, 2)
        for _ in range(N_PROBES):
            xg.grad = None
            (proj(xg)[sel] * torch.randn(len(sel), H)).sum().backward()
            g2 = xg.grad.reshape(-1, M).pow(2).sum(1)   # (frames,)
            for t in sel.tolist():                      # each frame belongs to exactly one probed token's field
                lo, hi = max(0, S * t - S + 1), min(len(mel), S * t + S)
                band[t, lo:hi] += g2[lo:hi]
band = (band / N_PROBES).sqrt()
est = band[j, f0:f0 + len(per_frame)]
print(f"probe estimate vs exact Jacobian for token {j}: {((est - per_frame).norm() / per_frame.norm()):.1%} relative error")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [2, 1]})
axes[0].imshow(band, aspect="auto", cmap="magma", interpolation="nearest", origin="lower",
               extent=[0, len(mel) / FPS, -0.5, n_tok - 0.5])
axes[0].set(xlabel="time (s)", ylabel="audio token", title="influence of every frame on every token")
z0, z1 = max(0, j - 5), min(n_tok, j + 6)
axes[1].imshow(band[z0:z1, S * z0:S * z1], aspect="auto", cmap="magma", interpolation="nearest", origin="lower",
               extent=[S * z0 - 0.5, S * z1 - 0.5, z0 - 0.5, z1 - 0.5])
for k in range(z0, z1 + 1):
    axes[1].axvline(k * S - 0.5, color="w", lw=0.4, alpha=0.5)
axes[1].axhline(j, color="C0", lw=0.8)
axes[1].set(xlabel="frame", ylabel="audio token", title=f"zoom around token {j} (white lines = patch edges)")
plt.tight_layout(); plt.show()

## 5. Where does the token land?

The token replaces a word embedding in the LLM's input, so compare it with the LLM's text embedding table, read
straight from the checkpoint's safetensors without loading the model. Three views:

* **Norms.** `PROJ_INIT = "match_emb"` starts audio tokens at text-embedding scale (row norm ~0.8 vs ~0.9). Has
  training kept them there? (Log scale: they usually haven't.)
* **Nearest vocabulary entries** (cosine) for token `j` and its neighbours. If the cosines are low, the projector has not
  learned to "spell" the audio as words. It writes into its own region, and the LoRA'd LLM learns to read it.
* **2-D PCA of directions** (unit-normalised vectors, so the large audio norms don't swamp the plot) of a vocabulary
  sample plus this utterance's tokens. The line follows the utterance through time.

In [ ]:
import glob
from huggingface_hub import hf_hub_download
from safetensors import safe_open

model_dir = Path(hf_hub_download(T.MODEL, "config.json")).parent
for f in glob.glob(str(model_dir / "*.safetensors")):
    with safe_open(f, "pt") as st:
        for key in st.keys():
            if "embed_tokens" in key:
                emb = st.get_tensor(key).float()
print("text embeddings", tuple(emb.shape))

emb_n, y_n = F.normalize(emb, dim=1), F.normalize(y, dim=1)
sims = y_n @ emb_n.T                                   # (n_tok, vocab)
ratio = y.norm(dim=1).median() / emb.norm(dim=1).median()
print(f"median norm: audio tokens {y.norm(dim=1).median():.1f}, text embeddings {emb.norm(dim=1).median():.2f} "
      f"-> audio tokens are {ratio:.0f}x larger")
for t in [t for t in (j - 1, j, j + 1) if 0 <= t < n_tok]:
    top = sims[t].topk(8)
    words = ", ".join(f"{tok.decode([i])!r} {v:.2f}" for v, i in zip(top.values.tolist(), top.indices.tolist()))
    print(f"token {t:>3}  |y| = {y[t].norm():.2f}  nearest vocab: {words}")

rng = torch.Generator().manual_seed(0)
vocab_ids = torch.randperm(len(emb), generator=rng)[:4000]
vocab_nn = torch.cat([(emb_n[ids] @ emb_n.T).scatter_(1, ids[:, None], -1).max(1).values   # nearest *other* word
                      for ids in vocab_ids.split(500)])
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
bins = np.logspace(np.log10(0.3), np.log10(float(y.norm(dim=1).max()) * 1.2), 70)
axes[0].hist(emb.norm(dim=1), bins=bins, alpha=0.6, color="0.5", label="vocab", density=True)
axes[0].hist(y.norm(dim=1), bins=bins, alpha=0.8, color="C0", label="audio tokens", density=True)
axes[0].set(title="L2 norm (log scale)", xlabel="norm", xscale="log", yticks=[]); axes[0].legend(fontsize=8)
axes[1].hist(vocab_nn, bins=50, density=True, alpha=0.6, color="0.5", label="word → nearest other word")
axes[1].hist(sims.max(1).values, bins=30, density=True, alpha=0.8, color="C0", label="audio token → nearest word")
axes[1].set(title="cosine to the nearest vocabulary entry", xlabel="cosine"); axes[1].legend(fontsize=8)

both = torch.cat([emb_n[vocab_ids], y_n])
mu = both.mean(0)
_, _, V = torch.pca_lowrank(both - mu, q=2, center=False)
pv, py = (emb_n[vocab_ids] - mu) @ V, (y_n - mu) @ V
axes[2].scatter(pv[:, 0], pv[:, 1], s=3, color="0.75", label="vocab sample")
axes[2].plot(py[:, 0], py[:, 1], "-", color="C0", lw=0.8, alpha=0.7)
axes[2].scatter(py[:, 0], py[:, 1], s=10, c=np.arange(n_tok), cmap="viridis", zorder=3, label="audio tokens (colour = time)")
axes[2].scatter(py[j, 0], py[j, 1], s=90, facecolor="none", edgecolor="C3", lw=2, zorder=4, label=f"token {j}")
axes[2].set(title="PCA of directions: vocab + this utterance", xlabel="PC1", ylabel="PC2"); axes[2].legend(fontsize=7, markerscale=2)
plt.tight_layout(); plt.show()

## 6. What does the first conv look for?

`conv1` is the only layer that reads mel bins directly, so its 512 kernels can be drawn as small spectrogram patches:
`80 mel bins × 3 frames`, red = positive weight, blue = negative. The panels show the 16 kernels that fire most at
token `j` (mean over the conv1 positions in its tree). Below them: how much weight each of the 3 taps carries, and
which mel bins the layer listens to.

In [ ]:
W1 = proj.convs[0].weight.view(-1, M, 3)               # (512, N_MELS, taps: previous / centre / next frame)
fam1 = sorted(tree["conv1"])
top = acts["conv1"][fam1].mean(0).argsort(descending=True)[:16]

fig = plt.figure(figsize=(14, 6.5))
fig.subplots_adjust(left=0.05, right=0.98, top=0.86, bottom=0.08, wspace=0.35, hspace=0.35)
for k, ch in enumerate(top.tolist()):
    ax = fig.add_subplot(2, 16, k + 1)
    lim = W1[ch].abs().max()
    ax.imshow(W1[ch], origin="lower", aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([] if k else [0, 40, 79])
    ax.set_title(f"#{ch}\n{acts['conv1'][fam1, ch].mean():.1f}", fontsize=7)
fig.text(0.05, 0.95, f"conv1 kernels most active at token {j} (title: channel, mean activation); y = mel bin, x = 3 frames",
         fontsize=9)
ax = fig.add_subplot(2, 3, 4)
ax.bar(["previous", "centre", "next"], W1.abs().mean((0, 1)), color="C0", width=0.6)
ax.set(title="mean |weight| per tap", ylabel="|w|")
ax = fig.add_subplot(2, 3, (5, 6))
ax.plot(W1.norm(dim=2).mean(0), color="C0", lw=2)
ax.set(title="mean kernel norm per mel bin: which frequencies conv1 listens to", xlabel="mel bin")
plt.show()

## Takeaways

* **One token = 80 ms + 70 ms of look-back.** The conv stack gives every token a 15-frame (150 ms) window: its own patch
  plus the previous patch minus its first frame (sections 2 and 4). Neighbouring tokens overlap. Nothing looks ahead
  past the end of its own patch.
* **Time mixing happens only in the convs.** After `conv3` there is exactly one position per token. LayerNorm, `fc1` and
  `out` act on each token alone, so any context beyond 150 ms has to come from the LLM's attention and short-conv blocks
  (with LoRA).
* **The architecture shapes what training learns.** In section 4a the learned per-frame influence follows the tree's
  1-3-path comb closely. Frames that only one path reaches matter less, whatever they contain. On the default sample
  (`mix544h_conv_stack8`, token 47), about half of the token's sensitivity comes from the *previous* patch.
* **The units are sparse.** Only ~10-15 % of the units after `conv2`, `conv3` and `fc1` are clearly on for a token (3b).
* **Audio tokens are not words, and they are much louder.** Their nearest vocabulary entry has cosine ~0.1-0.15, where
  real words are 0.3-0.9 from each other. Their norms grew from ~0.8 at init to ~30-160, against ~0.9 for text
  embeddings. The LLM is adapted (LoRA) to read them anyway, but tokens this large will dominate the residual stream
  at the audio positions. A norm penalty or an output LayerNorm on the projector would be a cheap experiment.

To look at another token or utterance, change `TOKEN` / `SAMPLE_INDEX` in section 0 and *Run All*. To look at another
run, set `RUN`. A `STACK = 16` conv run works the same way, with 4 conv stages and a 31-frame window.